# A6 Leakage-Safe Split (CPU, LOCKED)

Membuat split `train`/`validation`/`test` yang *leakage-safe* dari silver labels
melalui `sipature_ml.split.run_split`. Ikuti `docs/leakage-safe-split-baseline-report.md`
dan `docs/reproducibility-runbook.md` sebelum eksekusi.

Input: `data/processed/canonical_reviews.parquet` (notebook `02`) dan
`data/annotations/silver-v1.0.0.jsonl` (notebook `03`).
Output: `data/splits/*.jsonl` + `split_manifest_silver_v1.json` (terkunci).

**PENTING (locked-test policy):** split hanya boleh dibuat SEKALI. Notebook ini
menolak membuat ulang bila manifest sudah ada di Drive, dan test split tidak
boleh dibaca sebelum model/threshold dibekukan (A8).


## Step 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Konfigurasi path & parameter


In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"
DRIVE_ANNOTATION_DIR = DRIVE_ROOT / "data" / "annotations"
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"

PROJECT_DIR = Path("/content/hackathon/ml")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ANNOTATION_DIR = PROJECT_DIR / "data" / "annotations"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"

DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"

print("Drive root:", DRIVE_ROOT)
print("Sumber canonical reviews:", DRIVE_PROCESSED_DIR / "canonical_reviews.parquet")
print("Sumber silver labels   :", DRIVE_ANNOTATION_DIR / "silver-v1.0.0.jsonl")
print("Split dir (lokal)      :", SPLIT_DIR)


## Step 3 — Clone repository dari GitHub


In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


## Step 4 — Verifikasi commit terbaru (git log)


In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


## Step 5 — Install dependencies


In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


## Step 6 — Verifikasi versi package


In [ ]:
import numpy
import pandas
import pyarrow
import sklearn

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)


## Step 7 — Copy input dari Drive (canonical + silver)


In [ ]:
# Salin input (canonical_reviews + silver labels) dari Drive ke lokal.
import shutil
from pathlib import Path

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

inputs = [
    (DRIVE_PROCESSED_DIR / "canonical_reviews.parquet", PROCESSED_DIR / "canonical_reviews.parquet"),
    (DRIVE_ANNOTATION_DIR / "silver-v1.0.0.jsonl", ANNOTATION_DIR / "silver-v1.0.0.jsonl"),
]

for source, destination in inputs:
    assert source.is_file(), (
        f"Input tidak ditemukan di Drive: {source}\n"
        "Jalankan notebook 02 dan 03 terlebih dahulu."
    )
    shutil.copy2(source, destination)
    print("Disalin:", source.name, "->", destination)


## Step 8 — Import modul sipature_ml


In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


## Step 9 — Load split config & validasi silver


In [ ]:
from sipature_ml.config import load_config
from sipature_ml.annotation import validate_silver_records

split_config = load_config("split")

print("Split version :", split_config["split_version"])
print("Seed          :", split_config["seed"])
print("Group column  :", split_config["group_column"])
print("Ratios        :", split_config["ratios"])
print("Annotation file :", split_config["annotation_file"])
print("Canonical file  :", split_config["canonical_reviews_file"])
print("Algorithm candidates:", split_config["algorithm"]["candidates"])

# Quality gate: validasi silver JSONL sebelum split.
validation = validate_silver_records(ANNOTATION_DIR / split_config["annotation_file"])
print("\nSilver validation -> records:", validation["records"],
      "| invalid:", validation["invalid_records"])
assert validation["invalid_records"] == 0, f"Silver invalid: {validation['errors']}"
print("Silver JSONL valid.")


## Step 10 — Guard split (buat / salin yang terkunci)


In [ ]:
# Guard locked split: jangan buat ulang bila manifest sudah ada di Drive.
import shutil
from pathlib import Path

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

drive_manifest = DRIVE_SPLIT_DIR / "split_manifest_silver_v1.json"

if drive_manifest.is_file():
    print("Split SUDAH terkunci di Drive. Menyalin split yang ada, TANPA membuat ulang.")
    for source in sorted(DRIVE_SPLIT_DIR.glob("*")):
        if source.is_file():
            shutil.copy2(source, SPLIT_DIR / source.name)
            print("Disalin dari Drive:", source.name)
else:
    from sipature_ml.split import run_split
    split_manifest = run_split(PROCESSED_DIR, ANNOTATION_DIR, SPLIT_DIR, REPORT_DIR)
    print("Split berhasil dibuat dan terkunci.")
    for split in ("train", "validation", "test"):
        dist = split_manifest["distribution"][split]
        print(f"  {split}: {dist['records']} records, {dist['destinations']} destinations")


## Step 11 — Tampilkan manifest split (distribution & leakage)


In [ ]:
# Tampilkan manifest split (distribution + leakage).
import json
from pathlib import Path

manifest_path = SPLIT_DIR / "split_manifest_silver_v1.json"
assert manifest_path.is_file(), "Split manifest tidak ditemukan"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

print("Split version:", manifest["split_version"])
print("Reference label:", manifest["reference_label_type"])
print("Test is locked:", manifest["test_is_locked"])
print("Component count:", manifest["component_count"])
print("Multi-destination components:", manifest["multi_destination_components"])
print("Cross-destination repeated-text groups:", manifest["cross_destination_repeated_text_groups"])

print("\nDISTRIBUTION:")
for split in ("train", "validation", "test"):
    dist = manifest["distribution"][split]
    print(f"  {split}: {dist['records']} records, {dist['destinations']} destinations, "
          f"{dist['empty_label_records']} empty-label")

print("\nLEAKAGE (semua harus 0):")
print(json.dumps(manifest["validation"]["leakage_counts"], indent=2))


## Step 12 — Copy output ke Drive


In [ ]:
# Salin split + manifest + report ke Drive (artefak persisten & terkunci).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (SPLIT_DIR, DRIVE_SPLIT_DIR),
    (REPORT_DIR, DRIVE_REPORT_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file() and source.name not in {"README.md"}:
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


## Step 13 — Run summary (hash & locked-test reminder)


In [ ]:
# ============================================================
# RUN SUMMARY — hash, path, dan locked-test reminder.
# ============================================================
from sipature_ml.manifest import sha256_file

print("SPLIT VERSION:", manifest["split_version"])
print("SEED:", manifest["seed"])
print("RATIOS:", manifest["ratios"])

print("\nSOURCE HASHES:")
print("  silver_sha256            :", manifest["sources"]["silver_sha256"])
print("  canonical_reviews_sha256 :", manifest["sources"]["canonical_reviews_sha256"])
print("  split_config_sha256      :", manifest["sources"]["split_config_sha256"])
print("  taxonomy_sha256          :", manifest["sources"]["taxonomy_sha256"])

print("\nOUTPUT SPLIT DIR  :", SPLIT_DIR)
print("DRIVE SPLIT DIR    :", DRIVE_SPLIT_DIR)
print("OUTPUT REPORT DIR  :", REPORT_DIR)

print("\nOUTPUT HASHES:")
for split, info in manifest["outputs"].items():
    print(f"  {info['path']}: {info['sha256']}")

print("\nREMINDER: test split TERKUNCI. Jangan dibaca sampai model & threshold dibekukan (A8).")
